In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization,
    ReLU, Add, GlobalAveragePooling1D, Dense
)
from tensorflow.keras.models import Model



2026-04-08 22:48:13.256379: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-08 22:48:13.266520: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775663293.278025  295461 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775663293.281902  295461 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775663293.291406  295461 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
pathload = "/home/dani/Documents/tugas akhir/TugasAkhirku2026/mrtcn_ids/Preprocessing/hasil/final/preprocessingimgsize256_s16/preprocessingimgsize256_s16.npz"
data = np.load(
    pathload,
    mmap_mode="r"   # penting: tidak load penuh ke RAM
)

print(data.files)

['x_train', 'y_train', 'x_val', 'y_val', 'x_test', 'y_test']


In [3]:
data = np.load(
    pathload,
)

X_train = data["x_train"]
y_train = data["y_train"]
X_val   = data["x_val"]
y_val   = data["y_val"]
X_test  = data["x_test"]
y_test  = data["y_test"]

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)


(13163, 256, 256) (13163,)
(5642, 256, 256) (5642,)
(12365, 256, 256) (12365,)


In [4]:
np.unique(y_train, return_counts=True)

(array([0, 1]), array([5337, 7826]))

In [5]:
import numpy as np

# Fungsi untuk cek balance
def check_balance_binary(y):
    n_normal = np.sum(y == 0)
    n_attack = np.sum(y == 1)
    total = len(y)
    print(f"Total samples: {total}")
    print(f"Normal: {n_normal} ({n_normal/total*100:.2f}%)")
    print(f"Attack: {n_attack} ({n_attack/total*100:.2f}%)\n")

# Cek balance di masing-masing split
print("=== Train Set ===")
check_balance_binary(y_train)

print("=== Validation Set ===")
check_balance_binary(y_val)

print("=== Test Set ===")
check_balance_binary(y_test)


=== Train Set ===
Total samples: 13163
Normal: 5337 (40.55%)
Attack: 7826 (59.45%)

=== Validation Set ===
Total samples: 5642
Normal: 2287 (40.54%)
Attack: 3355 (59.46%)

=== Test Set ===
Total samples: 12365
Normal: 6396 (51.73%)
Attack: 5969 (48.27%)



In [6]:
def tcn_block(x, filters, kernel_size, dilation):
    shortcut = x

    x = Conv1D(filters, kernel_size,
               padding="causal",
               dilation_rate=dilation)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv1D(filters, kernel_size,
               padding="causal",
               dilation_rate=dilation)(x)
    x = BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(filters, 1, padding="same")(shortcut)

    x = Add()([x, shortcut])
    x = ReLU()(x)
    return x


In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dropout, MaxPooling1D, Flatten, Dense

def build_mr_tcn(input_shape=(256, 256), dropout_rate=0.00):
    model = Sequential()

    # Conv1D #1
    model.add(Conv1D(
        filters=64,
        kernel_size=3,
        strides=1,
        dilation_rate=1,
        padding="same",
        activation="relu",
        input_shape=input_shape
    ))
    model.add(Dropout(dropout_rate))

    # Conv1D #2
    model.add(Conv1D(
        filters=64,
        kernel_size=3,
        strides=1,
        dilation_rate=2,
        padding="same",
        activation="relu"
    ))
    model.add(Dropout(dropout_rate))

    # Conv1D #3
    model.add(Conv1D(
        filters=64,
        kernel_size=3,
        strides=1,
        dilation_rate=4,
        padding="same",
        activation="relu"
    ))
    model.add(Dropout(dropout_rate))

    # MaxPooling
    model.add(MaxPooling1D(pool_size=8, strides=8))

    # Flatten
    model.add(Flatten())

    # Dropout
    model.add(Dropout(dropout_rate))

    # Linear → Sigmoid
    model.add(Dense(1, activation="sigmoid"))

    return model


In [8]:
model = build_mr_tcn()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


/home/dani/miniconda3/envs/latihan1/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1775663307.094988  295461 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4144 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 256, 64)        │        49,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 256, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 256, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 32, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         2,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,969 (296.75 KB)

 Trainable params: 75,969 (296.75 KB)

 Non-trainable params: 0 (0.00 B)

 Total params: 75,969 (296.75 KB)


In [9]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=30,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_mr_tcn.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)


In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=700,
    batch_size=8,
    callbacks=[early_stop, checkpoint]
)



Epoch 1/700


I0000 00:00:1775663310.720441  295743 service.cc:152] XLA service 0x7b7438004510 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775663310.720464  295743 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6
2026-04-08 22:48:30.745810: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775663310.887580  295743 cuda_dnn.cc:529] Loaded cuDNN version 91701
2026-04-08 22:48:31.277108: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.03GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-04-08 22:48:31.327664: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_b

 62/823 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4243 - loss: 0.7089

I0000 00:00:1775663311.796979  295743 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


818/823 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5490 - loss: 0.6815

2026-04-08 22:48:34.208388: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.03GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


823/823 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5492 - loss: 0.6815

2026-04-08 22:48:45.593341: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.38GiB (rounded to 1479016448)requested by op _EagerConst
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2026-04-08 22:48:45.593359: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1058] BFCAllocator dump for GPU_0_bfc
2026-04-08 22:48:45.593363: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (256): 	Total Chunks: 116, Chunks in use: 116. 29.0KiB allocated for chunks. 29.0KiB in use in bin. 2.7KiB client-requested in use in bin.
2026-04-08 22:48:45.593367: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (512): 	Total Chunks: 1, Chunks in use: 0. 512B allocated for chunks. 0B in use in bin. 0B client-requested in use in bin.
2026-04-08 22:4

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [ ]:
test_results = model.evaluate(X_test, y_test, verbose=0)

for name, val in zip(model.metrics_names, test_results):
    print(f"{name}: {val:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.legend()
plt.title("Loss")
plt.show()


In [ ]:
# ===== EVALUASI MODEL (TEST SET) =====

import numpy as np
from sklearn.metrics import roc_curve, confusion_matrix, classification_report

# 1. Ambil probabilitas output model
y_prob = model.predict(X_test).ravel()

# 2. Hitung ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# 3. Hitung Youden Index
youden_index = tpr - fpr
best_threshold = thresholds[np.argmax(youden_index)]

print("Best threshold (Youden):", best_threshold)

# 4. Prediksi final menggunakan threshold Youden
y_pred = (y_prob >= best_threshold).astype(int)

# 5. Evaluasi
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Normal", "Attack"]))


In [ ]:
y_pred = (y_prob >= best_threshold)


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=["Normal","Attack"]))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report

# 1. DEFINISI FUNGSI (Mesin Evaluasi)
def evaluate_iov_model(model, X_test, y_test):
    print("--- Memulai Proses Evaluasi ---")
    
    # Prediksi
    start_time = time.time()
    y_prob = model.predict(X_test).ravel()
    inference_time = (time.time() - start_time) / len(X_test)
    
    # Mencari Threshold Terbaik (Youden Index)
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    youden_index = tpr - fpr
    best_threshold = thresholds[np.argmax(youden_index)]
    
    # Klasifikasi Berdasarkan Threshold
    y_pred = (y_prob >= best_threshold).astype(int)
    
    # Hitung Metrik
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr_val = fp / (fp + tn)
    
    # VISUALISASI
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    
    # Heatmap Confusion Matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0])
    ax[0].set_title('Confusion Matrix (Optimized Threshold)')
    ax[0].set_xlabel('Predicted')
    ax[0].set_ylabel('Actual')
    
    # ROC Curve
    ax[1].plot(fpr, tpr, label=f'AUC = {auc(fpr, tpr):.44f}')
    ax[1].plot([0, 1], [0, 1], 'k--')
    ax[1].set_title('ROC Curve')
    ax[1].legend()
    
    plt.show() # Ini akan memunculkan gambar
    
    # CETAK HASIL KE KONSOL
    print("\n" + "="*30)
    print(f"HASIL EVALUASI IDS (IoV)")
    print("="*30)
    print(f"Threshold Optimal : {best_threshold:.4f}")
    print(f"False Positive Rate: {fpr_val:.6f}")
    print(f"Inference Latency : {inference_time:.6f} s/sample")
    print("-" * 30)
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))
    
    return best_threshold

# ==========================================
# 2. EKSEKUSI FUNGSI (Panggil di sini)
# ==========================================
# Ganti 'model_anda', 'X_test_anda', 'y_test_anda' 
# dengan nama variabel yang Anda miliki.

try:
    # Contoh: jika model Anda bernama 'global_model'
    best_thr = evaluate_iov_model(model, X_test, y_test)
except NameError as e:
    print(f"Error: {e}. Pastikan variabel model dan data Anda sudah didefinisikan sebelumnya.")

In [ ]:
from sklearn.metrics import confusion_matrix

# 1. Dapatkan prediksi dari model (berupa probabilitas)
y_prob = model.predict(X_test) 

# 2. Ubah probabilitas menjadi kelas (0 atau 1) 
# Gunakan threshold, misalnya 0.5 atau hasil Youden Index
y_pred = (y_prob > best_threshold).astype(int)

# 3. Munculkan Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# 4. Ekstrak TP, FP, TN, FN dari matriks
# Matriks sklearn memiliki format: [[TN, FP], [FN, TP]]
tn, fp, fn, tp = cm.ravel()

# 5. Cetak ke layar
print(f"--- Hasil Deteksi Serangan ---")
print(f"True Negatives (TN)  : {tn} (Trafik normal terdeteksi normal)")
print(f"False Positives (FP) : {fp} (ALARM PALSU - Normal dianggap serangan)")
print(f"False Negatives (FN) : {fn} (LOLOS - Serangan dianggap normal)")
print(f"True Positives (TP)  : {tp} (Berhasil mendeteksi serangan)")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', 
            xticklabels=['Normal', 'Attack'], 
            yticklabels=['Normal', 'Attack'])
plt.title('Confusion Matrix: Deteksi Serangan IoV')
plt.ylabel('Kondisi Aktual')
plt.xlabel('Prediksi Model')
plt.show()

batas


In [ ]:
import numpy as np

y_prob = model.predict(X_test).ravel()      # probabilitas Attack
y_pred = (y_prob > best_threshold).astype(int)         # label prediksi (threshold default)


In [ ]:
# Fungsi: deteksi overfitting dan stabilitas training.
import matplotlib.pyplot as plt

plt.figure()
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.show()

plt.figure()
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.show()


In [ ]:
# Fungsi: melihat False Negative (serangan lolos).
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Attack"]
)
disp.plot()
plt.title("Confusion Matrix")
plt.show()
# 📌 Interpretasi IDS

# FN (Attack → Normal) harus minimal

# FP tinggi → false alarm

In [ ]:
# Fungsi: menilai kualitas deteksi Attack.
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average=None
)

labels = ["Normal", "Attack"]

plt.figure()
plt.bar(labels, precision, label="Precision")
plt.bar(labels, recall, bottom=precision, label="Recall")
plt.ylabel("Score")
plt.title("Precision & Recall per Class")
plt.legend()
plt.show()
# Fokus laporan: Recall kelas Attack


In [ ]:
# Fungsi: evaluasi threshold-independent.
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# AUC mendekati 1 → IDS sangat kuat


In [ ]:
# Lebih relevan dari ROC untuk IDS.
from sklearn.metrics import precision_recall_curve

prec, rec, _ = precision_recall_curve(y_test, y_prob)

plt.figure()
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.show()


In [ ]:
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# ASUMSI:
# - y_prob SUDAH dihitung di evaluate_iov_model
# - best_thr didapat dari return fungsi
# ==========================================

# Gunakan ulang y_prob & y_test (TIDAK buat baru)
y_pred = (y_prob >= best_thr).astype(int)

# Hitung F1-score untuk threshold yang SAMA
f1 = f1_score(y_test, y_pred)

print("\n" + "="*30)
print("F1-SCORE (FOLLOW YOUDEN THRESHOLD)")
print("="*30)
print(f"Threshold digunakan : {best_thr:.4f}")
print(f"F1-score           : {f1:.6f}")


In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(best_thr, f1, color='red', zorder=5)
plt.axvline(best_thr, linestyle='--', label=f"Threshold = {best_thr:.4f}")

plt.text(
    best_thr, f1,
    f"({best_thr:.4f}, {f1:.4f})",
    ha='left', va='bottom'
)

plt.xlabel("Threshold")
plt.ylabel("F1-score")
plt.title("F1-score at Optimized Youden Threshold")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Fungsi: lihat separasi Normal vs Attack.
plt.figure()
plt.hist(y_prob[y_test == 1], bins=50, alpha=0.7, label="Attack")
plt.hist(y_prob[y_test == 0], bins=50, alpha=0.7, label="Normal")
plt.xlabel("Predicted Probability (Attack)")
plt.ylabel("Frequency")
plt.title("Prediction Confidence Distribution")
plt.legend()
plt.show()
# 📌 Distribusi terpisah jelas → model robust.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test, y_pred,
    target_names=["Normal", "Attack"]
))


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

def compute_ids_metrics(y_true, y_pred):
    """
    Compute IDS metrics:
    Accuracy, Precision, Recall, F1, FPR, FNR
    """

    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Basic metrics
    acc = (tp + tn) / (tp + tn + fp + fn)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    # IDS-specific metrics
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "FPR": fpr,
        "FNR": fnr,
    }

In [ ]:

# Hitung metrik IDS
metrics = compute_ids_metrics(y_test, y_pred)

for k, v in metrics.items():
    print(f"{k:10s}: {v:.6f}" if isinstance(v, float) else f"{k:10s}: {v}")


TP        : 10617
FP        : 704
TN        : 12463
FN        : 950
Accuracy  : 0.933128
Precision : 0.937815
Recall    : 0.917870
F1-score  : 0.927735
FPR       : 0.053467
FNR       : 0.082130